# 01 · Build NTCIR-DDG from the public NTCIR-15 Data Search collection (Colab)

The AutoDDG paper ([arXiv 2502.01050](https://arxiv.org/abs/2502.01050)) evaluates on **NTCIR-DDG**, a CSV-only subset of
NTCIR-15 Data Search, but never released it. This notebook rebuilds it from
[`mpkato/ntcir_data_search`](https://huggingface.co/datasets/mpkato/ntcir_data_search) and runs every **LLM-free** step.

| Section | What it does | Writes (under `WORK_DIR` on Drive) |
|---|---|---|
| 0 | Configuration (everything tweakable lives here) | – |
| 1 | Setup: mount Drive, install, download the small files | `hf/` |
| 2 | Explore the raw benchmark | – (prints only) |
| 3 | Build NTCIR-DDG under several configurations, pick the one closest to the paper | `benchmark/` |
| 4 | Acquire the needed CSV files (source URLs and/or one pass over the 478 GB archive) | `acquisition/` (raw files go to `/content/raw`, **not** Drive) |
| 5 | Read + profile every table (AutoDDG profiler, 100-row sample) | `intermediate/tables.jsonl` |
| 6 | Statistics vs the paper, failures, coverage | `benchmark/*_acquired.*` |
| 7 | Package everything for the Mac | `ntcir_ddg_intermediate.zip` |

**Every heavy step is idempotent**: re-running a cell skips work whose output already exists, so after a Colab
disconnect you can simply *Run all* again.

The LLM steps (semantic profile, topic, UFD, SFD) run on the Mac (`mac/generate_descriptions.py`); retrieval
evaluation is notebook 02.

## 0. Configuration

* `FIXTURE_MODE=True` runs the whole notebook against the tiny synthetic fixture built by `tests/make_fixture.py`
  (no Hugging Face, no Drive). The env var `NTCIR_DDG_FIXTURE=1` does the same (used by the automated tests).
* Benchmark choices the paper does not specify (topic set, qrels, relevance threshold, CSV test) are explored in
  section 3; by default the closest configuration to the paper is picked automatically (`AUTO_SELECT_CONFIG`).

In [ ]:
import os

# ---------------- mode ----------------
FIXTURE_MODE = False
if os.environ.get("NTCIR_DDG_FIXTURE") == "1":
    FIXTURE_MODE = True
FIXTURE_DIR = os.environ.get("NTCIR_DDG_FIXTURE_DIR", "/content/fixture")

# ---------------- paths ----------------
WORK_DIR = os.environ.get("NTCIR_DDG_WORK_DIR", "/content/drive/MyDrive/ntcir_ddg")  # small outputs (Drive)
RAW_DIR = os.environ.get("NTCIR_DDG_RAW_DIR", "/content/raw")  # raw tables (local Colab disk only)

# ---------------- source ----------------
HF_REPO = "mpkato/ntcir_data_search"
HF_TOKEN = os.environ.get("HF_TOKEN")  # optional; raises HF rate limits
AUTODDG_SPEC = "autoddg @ https://github.com/VIDA-NYU/AutoDDG/archive/f286ac697bdb0a199eac1ce0778401f30341b4bb.zip"

# ---------------- benchmark construction (section 3) ----------------
MIN_RELEVANT = 5      # paper: queries with >= 5 relevant ...
MIN_CANDIDATES = 10   # ... and >= 10 judged CSV datasets
AUTO_SELECT_CONFIG = True  # False -> use MANUAL_CONFIG
MANUAL_CONFIG = {"topic_set": "ntcir15_train+test", "test_qrels": "test", "rel_min": 1, "csv_mode": "mime"}
# topic_set : ntcir15_test | ntcir15_train | ntcir15_train+test | ntcir16_train | ntcir16_test | ntcir16_train+test
# test_qrels: test | all   (only matters for NTCIR-15 test topics; 'all' = test + extra judgments)
# rel_min   : 1 -> relevant = L1+L2 ; 2 -> relevant = L2 only
# csv_mode  : mime (stored file is text/csv) | format (data_format says csv) | mime_or_format | mime_all
PAPER = {"n_queries": 32, "rel_avg": 10, "rel_min": 5, "rel_max": 22, "cand_avg": 19, "cand_min": 10,
         "cand_max": 42, "tables_total": 615, "rows_avg": 78630, "rows_min": 3, "rows_max": 2717008,
         "cols_avg": 14, "cols_min": 2, "cols_max": 74, "ndcg20_original": 0.688, "ndcg20_sfd": 0.705}
MULTI_FILE_POLICY = "first"  # first (first listed CSV that was acquired) | largest | concat (same header only)

# ---------------- acquisition (section 4) ----------------
ACQUISITION_STRATEGY = "auto"  # auto | url | archive
ARCHIVE_EXTRACT = "all"        # all: take every needed file from the 2019 archive snapshot (recommended)
                               # missing: only files the URL pass could not get
CONFIRM_ARCHIVE_STREAM = False # safety switch: the full archive pass takes hours; set True to start it
URL_CHECK_SAMPLE = 50
URL_TIMEOUT = 60               # seconds per socket read
URL_RETRIES = 3
URL_WORKERS = 8
URL_MAX_BYTES = 3_000_000_000
PEEK_MEMBERS = 300

# ---------------- preprocessing (section 5) ----------------
SAMPLE_SIZE = 100              # rows in the sample the LLM sees (quick-start notebook)
SAMPLE_SEED = 9                # autoddg.utils.get_sample default random_state
LARGE_FILE_BYTES = 100_000_000 # above this, stream the file: exact row count + reservoir sample
PROFILE_SAMPLE_ROWS = 200_000  # reservoir size for large files (datamart samples to 5 MB anyway)
READ_CHUNK_ROWS = 100_000
RESERVOIR_SEED = 0
RETRY_FAILED_TABLES = False    # re-profile tables whose previous attempt errored (not-acquired ones always retry)
DELETE_RAW_AFTER_PROFILE = False

if FIXTURE_MODE:  # small knobs so every code path runs in seconds
    WORK_DIR = os.environ.get("NTCIR_DDG_WORK_DIR", "/content/fixture_work")
    RAW_DIR = os.environ.get("NTCIR_DDG_RAW_DIR", "/content/fixture_raw")
    LARGE_FILE_BYTES = 200_000
    PROFILE_SAMPLE_ROWS = 5_000
    READ_CHUNK_ROWS = 2_000
    URL_TIMEOUT = 2
    URL_RETRIES = 2
    URL_CHECK_SAMPLE = 10
    PEEK_MEMBERS = 10
    CONFIRM_ARCHIVE_STREAM = True
# Optional JSON overrides of any variable above, e.g. NTCIR_DDG_OVERRIDES='{"MULTI_FILE_POLICY": "concat"}'
# (used by the automated tests; on Colab just edit the values above)
if os.environ.get("NTCIR_DDG_OVERRIDES"):
    import json as _json

    globals().update(_json.loads(os.environ["NTCIR_DDG_OVERRIDES"]))
print(f"FIXTURE_MODE={FIXTURE_MODE}  WORK_DIR={WORK_DIR}  RAW_DIR={RAW_DIR}")

## 1. Setup

Mounts Drive (only on Colab), installs the pinned AutoDDG commit (PyPI `autoddg==0.1.2` lacks
`profile_metadata(..., nb_rows=...)` and `summarize`, which we need for large files), `lbzip2` (parallel bzip2) and `pv`.
If pip reports that the runtime must be restarted, do *Runtime → Restart session* and run all cells again.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and WORK_DIR.startswith("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
if IN_COLAB:
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y lbzip2 pv > /dev/null", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", AUTODDG_SPEC, "huggingface_hub"], check=True)

In [ ]:
import bz2
import codecs
import collections
import csv
import gzip
import hashlib
import io
import json
import math
import random
import re
import shutil
import statistics
import tarfile
import threading
import time
import warnings
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from importlib.metadata import version as pkg_version
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", message="transformers and torch not available")
import autoddg  # noqa: E402
from autoddg.profiling import profile_dataset, profile_metadata, summarize  # noqa: E402
from autoddg.utils import get_sample  # noqa: E402

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

WORK = Path(WORK_DIR)
HF_DIR, BENCH, INTER, ACQ = WORK / "hf", WORK / "benchmark", WORK / "intermediate", WORK / "acquisition"
RAW = Path(RAW_DIR)
for p in (HF_DIR, BENCH, INTER, ACQ, RAW / "archive", RAW / "url"):
    p.mkdir(parents=True, exist_ok=True)

DECOMPRESSOR = shutil.which("lbzip2") or shutil.which("pbzip2") or shutil.which("bzip2")
for pkg in ["pandas", "numpy", "autoddg", "datamart_profiler", "huggingface_hub", "requests"]:
    try:
        print(f"{pkg:18s} {pkg_version(pkg)}")
    except Exception:
        print(f"{pkg:18s} (not installed)")
print(f"{'python':18s} {sys.version.split()[0]}")
print(f"{'decompressor':18s} {DECOMPRESSOR}   pv: {shutil.which('pv')}")
total, used, free = shutil.disk_usage(RAW)
print(f"local disk free     {free / 1e9:.1f} GB")


def log(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def append_jsonl(path: Path, row: dict) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:  # a line cut by a crash mid-write
                    pass
    return rows

### 1b. Download the small files
Everything except the 478 GB data archive (≈ 30 MB total). Skipped if already on Drive.

In [ ]:
SMALL_FILES = [
    "README.md",
    "data_search_e_collection.jsonl.bz2",
    "data_search_e_train_topics.tsv", "data_search_e_test_topics.tsv",
    "data_search_e_train_qrels.txt", "data_search_e_test_qrels.txt",
    "data_search_e_extra_qrels.txt", "data_search_e_all_qrels.txt",
    "data_search_2_e_train_topics.tsv", "data_search_2_e_test_topics.tsv",
    "data_search_2_e_train_qrels.txt", "data_search_2_e_test_qrels.txt",
]
ARCHIVE_NAME = "data_search_e_data.tar.bz2"
ARCHIVE_URL = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{ARCHIVE_NAME}"
ARCHIVE_SIZE = 477_889_927_989  # bytes, from the HF file listing

if FIXTURE_MODE:
    ARCHIVE_PATH = Path(FIXTURE_DIR) / "hf" / ARCHIVE_NAME
    ARCHIVE_SIZE = ARCHIVE_PATH.stat().st_size
else:
    from huggingface_hub import hf_hub_download

for fn in SMALL_FILES:
    dst = HF_DIR / fn
    if dst.exists() and dst.stat().st_size > 0:
        continue
    if FIXTURE_MODE:
        shutil.copy(Path(FIXTURE_DIR) / "hf" / fn, dst)
    else:
        hf_hub_download(HF_REPO, fn, repo_type="dataset", local_dir=str(HF_DIR), token=HF_TOKEN)
for fn in SMALL_FILES:
    print(f"{fn:42s} {(HF_DIR / fn).stat().st_size:>12,d} bytes")

### 1c. (fixture mode only) serve the fake source URLs
The fixture's `data_url`s point at `http://127.0.0.1:18765`. URLs under `/slow/` stall longer than `URL_TIMEOUT`.

In [ ]:
if FIXTURE_MODE:
    import functools
    import http.server

    class _FixtureHandler(http.server.SimpleHTTPRequestHandler):
        def do_GET(self):  # noqa: N802
            if self.path.startswith("/slow/"):
                time.sleep(URL_TIMEOUT + 1.5)
                self.path = "/files/" + self.path[len("/slow/"):]
            return super().do_GET()

        def log_message(self, *args):
            pass

        def copyfile(self, source, outputfile):
            try:
                super().copyfile(source, outputfile)
            except (BrokenPipeError, ConnectionResetError):
                pass  # client closed early (liveness check reads only 2 KB)

    class _QuietServer(http.server.ThreadingHTTPServer):
        def handle_error(self, request, client_address):
            pass

    try:
        _srv = _QuietServer(
            ("127.0.0.1", 18765),
            functools.partial(_FixtureHandler, directory=str(Path(FIXTURE_DIR) / "web")))
        threading.Thread(target=_srv.serve_forever, daemon=True).start()
        print("fixture URL server on http://127.0.0.1:18765")
    except OSError:
        print("fixture URL server already running")

## 2. Explore the raw benchmark

Nothing here is assumed: we print the schema, examples and distributions so we understand the data before
filtering it.

In [ ]:
collection = [json.loads(line) for line in bz2.open(HF_DIR / "data_search_e_collection.jsonl.bz2", "rt", encoding="utf-8")]
by_id = {r["id"]: r for r in collection}
print(f"datasets in the English collection: {len(collection):,d}  (unique ids: {len(by_id):,d})")


def type_name(v):
    return type(v).__name__


def nonempty(v):
    return v not in (None, "", [], {})


top_fill, top_types = collections.Counter(), collections.defaultdict(collections.Counter)
data_fill, data_types = collections.Counter(), collections.defaultdict(collections.Counter)
field_fill = collections.Counter()
n_items = 0
for r in collection:
    for k, v in r.items():
        top_types[k][type_name(v)] += 1
        top_fill[k] += nonempty(v)
    for d in r.get("data") or []:
        n_items += 1
        for k, v in d.items():
            data_types[k][type_name(v)] += 1
            data_fill[k] += nonempty(v)
    for k, v in (r.get("data_fields") or {}).items():
        field_fill[k] += nonempty(v)

print("\nTop-level keys (fill rate = non-empty share):")
display(pd.DataFrame([{"key": k, "fill_rate": f"{top_fill[k] / len(collection):.1%}", "types": dict(top_types[k])}
                      for k in top_types]))
print(f"\nKeys inside `data` items ({n_items:,d} items):")
display(pd.DataFrame([{"key": k, "fill_rate": f"{data_fill[k] / n_items:.1%}", "types": dict(data_types[k])}
                      for k in data_types]))
print(f"\n`data_fields` is a free-form dict with {len(field_fill)} distinct keys; the 30 most common:")
display(pd.DataFrame([{"key": k, "fill_rate": f"{c / len(collection):.1%}"} for k, c in field_fill.most_common(30)]))

In [ ]:
for r in [collection[0], collection[len(collection) // 2], collection[-1]]:
    print(json.dumps(r, indent=2, ensure_ascii=False))
    print("-" * 100)

### Files per dataset, formats, CSV share
`data_format` is the publisher's label; the suffix of `data_filename` is the MIME type of the file that was actually
stored in 2019 (`<sha256(data_url)>.<mime with / replaced by .>`). They disagree often (e.g. "csv" that is really an
HTML landing page), which is why section 3 compares several CSV tests.

In [ ]:
def mime_of(d):
    parts = d["data_filename"].split(".", 1)
    return parts[1] if len(parts) > 1 else "(none)"


def is_csv_mime(d):
    return d["data_filename"].lower().endswith(".text.csv")


def is_csv_format(d):
    return "csv" in (d.get("data_format") or "").lower()


CSV_FILE_TESTS = {
    "mime": is_csv_mime,
    "format": is_csv_format,
    "mime_or_format": lambda d: is_csv_mime(d) or is_csv_format(d),
    "mime_all": is_csv_mime,  # file test is the same; the dataset test differs (all files must be CSV)
}


def csv_files(rec, mode):
    test = CSV_FILE_TESTS[mode]
    return [d for d in rec.get("data") or [] if test(d)]


def is_csv_dataset(rec, mode):
    data = rec.get("data") or []
    if mode == "mime_all":
        return bool(data) and all(is_csv_mime(d) for d in data)
    return bool(csv_files(rec, mode))


nfiles = pd.Series([len(r.get("data") or []) for r in collection])
print("files per dataset:")
print(nfiles.describe().round(2).to_string())
print(nfiles.value_counts().sort_index().head(12).to_string())
fmt = collections.Counter((d.get("data_format") or "").lower() for r in collection for d in r["data"])
mime = collections.Counter(mime_of(d) for r in collection for d in r["data"])
_a = [f"{k} ({v:,d})" for k, v in fmt.most_common(15)]
_b = [f"{k} ({v:,d})" for k, v in mime.most_common(15)]
_n = max(len(_a), len(_b))
display(pd.DataFrame({"data_format": _a + [""] * (_n - len(_a)), "filename MIME suffix": _b + [""] * (_n - len(_b))}))
for mode in CSV_FILE_TESTS:
    n = sum(is_csv_dataset(r, mode) for r in collection)
    print(f"datasets that are CSV under csv_mode={mode!r:17s}: {n:6,d}  ({n / len(collection):.1%})")

### Fields that look like source URLs, and licenses

In [ ]:
url_fill, url_examples = collections.Counter(), collections.defaultdict(list)


def scan(prefix, obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            scan(f"{prefix}.{k}" if prefix else k, v)
    elif isinstance(obj, list):
        for v in obj:
            scan(prefix + "[]", v)
    elif isinstance(obj, str) and "http" in obj:
        url_fill[prefix] += 1
        if len(url_examples[prefix]) < 2:
            url_examples[prefix].append(obj[:160])


for r in collection:
    scan("", r)
display(pd.DataFrame([{"field": k, "values containing 'http'": c, "per dataset": f"{c / len(collection):.2f}",
                       "example": url_examples[k][0]} for k, c in url_fill.most_common(20)]))
HAS_SOURCE_URLS = url_fill.get("data[].data_url", 0) > 0
print(f"\nper-file source URL field `data[].data_url` present: {HAS_SOURCE_URLS}")

lic = collections.Counter()
for r in collection:
    m = re.search(r"is licensed under (.+?)(?: \(http|$)", r.get("attribution") or "")
    lic[m.group(1).strip() if m else "(unparsed)"] += 1
print("\nlicense distribution (parsed from `attribution`):")
print(pd.Series(lic).sort_values(ascending=False).head(15).to_string())

### Topics and qrels

In [ ]:
def load_topics(fn):
    out = {}
    for line in open(HF_DIR / fn, encoding="utf-8"):
        if line.strip():
            qid, q = line.rstrip("\n").split("\t", 1)
            out[qid] = q
    return out


def load_qrels(fn):
    out = collections.defaultdict(dict)
    for line in open(HF_DIR / fn, encoding="utf-8"):
        parts = line.split()
        if len(parts) == 3:
            out[parts[0]][parts[1]] = int(parts[2].lstrip("Ll"))
    return dict(out)


TOPIC_FILES = {
    "ntcir15_train": "data_search_e_train_topics.tsv", "ntcir15_test": "data_search_e_test_topics.tsv",
    "ntcir16_train": "data_search_2_e_train_topics.tsv", "ntcir16_test": "data_search_2_e_test_topics.tsv"}
QRELS_FILES = {
    "ntcir15_train": "data_search_e_train_qrels.txt", "ntcir15_test": "data_search_e_test_qrels.txt",
    "ntcir15_extra": "data_search_e_extra_qrels.txt", "ntcir15_all": "data_search_e_all_qrels.txt",
    "ntcir16_train": "data_search_2_e_train_qrels.txt", "ntcir16_test": "data_search_2_e_test_qrels.txt"}
topics = {k: load_topics(v) for k, v in TOPIC_FILES.items()}
qrels = {k: load_qrels(v) for k, v in QRELS_FILES.items()}

rows = []
for k, t in topics.items():
    lens = [len(q.split()) for q in t.values()]
    rows.append({"topic set": k, "queries": len(t), "words/query avg": round(statistics.mean(lens), 2),
                 "min": min(lens), "max": max(lens)})
display(pd.DataFrame(rows))
print("10 example queries:")
for qid, q in list(topics["ntcir15_test"].items())[:5] + list(topics["ntcir15_train"].items())[:5]:
    print(f"  {qid}: {q}")
print("query length (words) distribution, NTCIR-15 train+test:")
print(pd.Series([len(q.split()) for k in ("ntcir15_train", "ntcir15_test") for q in topics[k].values()])
      .value_counts().sort_index().to_string())

rows = []
for k, q in qrels.items():
    levels = collections.Counter(v for d in q.values() for v in d.values())
    judged = [len(d) for d in q.values()]
    rel = [sum(v >= 1 for v in d.values()) for d in q.values()]
    missing = sum(1 for d in q.values() for i in d if i not in by_id)
    rows.append({"qrels": k, "queries": len(q), "judgments": sum(judged), "L0": levels[0], "L1": levels[1],
                 "L2": levels[2], "judged/query avg": round(statistics.mean(judged), 1),
                 "judged min-max": f"{min(judged)}-{max(judged)}", "L1+L2/query avg": round(statistics.mean(rel), 1),
                 "doc ids not in collection": missing})
display(pd.DataFrame(rows))

## 3. Build NTCIR-DDG candidates under several configurations

For a query, its **candidates** are its judged datasets that pass the CSV test, and **relevant** means level
≥ `rel_min` among them. A query is kept if it has ≥ 5 relevant and ≥ 10 candidates (the paper's rule). We try every
combination of the choices the paper leaves open and rank them by their distance to the paper's statistics
(relative errors on #queries, mean relevant, mean candidates, total tables, plus a down-weighted term for the ranges).

"Total tables" is reported two ways: summed over queries (a table shared by two queries counts twice) and unique. The
paper's 615 ≈ 32 × 19 suggests the summed count, so that is the one used in the distance.

In [ ]:
TOPIC_SETS = {  # topic set -> list of (topics key, qrels key or "@15test")
    "ntcir15_test": [("ntcir15_test", "@15test")],
    "ntcir15_train": [("ntcir15_train", "ntcir15_train")],
    "ntcir15_train+test": [("ntcir15_train", "ntcir15_train"), ("ntcir15_test", "@15test")],
    "ntcir16_train": [("ntcir16_train", "ntcir16_train")],
    "ntcir16_test": [("ntcir16_test", "ntcir16_test")],
    "ntcir16_train+test": [("ntcir16_train", "ntcir16_train"), ("ntcir16_test", "ntcir16_test")],
}
TEST_QRELS = {"test": "ntcir15_test", "all": "ntcir15_all"}
CSV_DATASETS = {mode: {r["id"] for r in collection if is_csv_dataset(r, mode)} for mode in CSV_FILE_TESTS}


def build_benchmark(cfg):
    queries, bench_qrels, pools = {}, {}, {}
    csv_ids = CSV_DATASETS[cfg["csv_mode"]]
    for tkey, qkey in TOPIC_SETS[cfg["topic_set"]]:
        qk = TEST_QRELS[cfg["test_qrels"]] if qkey == "@15test" else qkey
        for qid, judged in qrels[qk].items():
            if qid not in topics[tkey]:
                continue
            cand = {d: lvl for d, lvl in judged.items() if d in csv_ids}
            n_rel = sum(lvl >= cfg["rel_min"] for lvl in cand.values())
            if n_rel >= MIN_RELEVANT and len(cand) >= MIN_CANDIDATES:
                queries[qid] = topics[tkey][qid]
                # graded relevance for NDCG: levels below rel_min count as 0
                bench_qrels[qid] = {d: (lvl if lvl >= cfg["rel_min"] else 0) for d, lvl in cand.items()}
                pools[qid] = sorted(cand)
    return queries, bench_qrels, pools


def bench_stats(queries, bench_qrels, pools):
    if not queries:
        return {"n_queries": 0}
    rel = [sum(v > 0 for v in bench_qrels[q].values()) for q in queries]
    cand = [len(pools[q]) for q in queries]
    return {"n_queries": len(queries), "rel_avg": statistics.mean(rel), "rel_min": min(rel), "rel_max": max(rel),
            "cand_avg": statistics.mean(cand), "cand_min": min(cand), "cand_max": max(cand),
            "tables_sum": sum(cand), "tables_unique": len({d for q in queries for d in pools[q]})}


def distance(s):
    if not s["n_queries"]:
        return float("inf")
    rel = lambda a, b: abs(a - b) / b  # noqa: E731
    return (rel(s["n_queries"], PAPER["n_queries"]) + rel(s["rel_avg"], PAPER["rel_avg"])
            + rel(s["cand_avg"], PAPER["cand_avg"]) + rel(s["tables_sum"], PAPER["tables_total"])
            + 0.25 * (rel(s["rel_min"], PAPER["rel_min"]) + rel(s["rel_max"], PAPER["rel_max"])
                      + rel(s["cand_min"], PAPER["cand_min"]) + rel(s["cand_max"], PAPER["cand_max"])))


grid = []
for ts, parts in TOPIC_SETS.items():
    uses_test = any(q == "@15test" for _, q in parts)
    for tq in (["test", "all"] if uses_test else ["test"]):
        for rel_min in (1, 2):
            for csv_mode in CSV_FILE_TESTS:
                cfg = {"topic_set": ts, "test_qrels": tq if uses_test else "-", "rel_min": rel_min, "csv_mode": csv_mode}
                s = bench_stats(*build_benchmark({**cfg, "test_qrels": tq}))
                grid.append({"cfg": cfg, "stats": s, "distance": distance(s)})
grid.sort(key=lambda g: g["distance"])
fmt_rng = lambda a, lo, hi: f"{a:.1f} ({lo}-{hi})"  # noqa: E731
show_rows = []
for g in grid:
    c, s = g["cfg"], g["stats"]
    show_rows.append({
        "topic_set": c["topic_set"], "test_qrels": c["test_qrels"],
        "relevant": "L1+L2" if c["rel_min"] == 1 else "L2", "csv_mode": c["csv_mode"], "#queries": s["n_queries"],
        "relevant/query avg (min-max)": fmt_rng(s["rel_avg"], s["rel_min"], s["rel_max"]) if s["n_queries"] else "-",
        "candidates/query avg (min-max)": fmt_rng(s["cand_avg"], s["cand_min"], s["cand_max"]) if s["n_queries"] else "-",
        "tables (sum over queries)": s.get("tables_sum", 0), "tables (unique)": s.get("tables_unique", 0),
        "distance": round(g["distance"], 3)})
grid_table = pd.DataFrame(show_rows)
print("paper: 32 queries | relevant/query 10 (5-22) | candidates/query 19 (10-42) | 615 tables")
display(grid_table.head(25))
grid_table.to_csv(BENCH / "config_grid.csv", index=False)

In [ ]:
best_cfg = dict(grid[0]["cfg"])
if best_cfg["test_qrels"] == "-":
    best_cfg["test_qrels"] = "test"
CHOSEN = best_cfg if AUTO_SELECT_CONFIG else dict(MANUAL_CONFIG)
print("closest to the paper:", best_cfg)
print("CHOSEN              :", CHOSEN, "(auto)" if AUTO_SELECT_CONFIG else "(manual override)")

queries, bench_qrels, pools = build_benchmark(CHOSEN)
stats = bench_stats(queries, bench_qrels, pools)
display(pd.DataFrame([{"": "ours", **{k: round(v, 2) if isinstance(v, float) else v for k, v in stats.items()}},
                      {"": "paper", "n_queries": 32, "rel_avg": 10, "rel_min": 5, "rel_max": 22, "cand_avg": 19,
                       "cand_min": 10, "cand_max": 42, "tables_sum": 615, "tables_unique": None}]).set_index(""))

### Three example queries
Relevant (level ≥ 1 after thresholding) vs non-relevant candidates, with their original titles and descriptions.

In [ ]:
def short(s, n=160):
    s = " ".join((s or "").split())
    return s if len(s) <= n else s[: n - 1] + "…"


rng_ex = random.Random(0)
for qid in rng_ex.sample(sorted(queries), min(3, len(queries))):
    print(f"\n=== {qid}: {queries[qid]!r}  ({len(pools[qid])} candidates)")
    ranked = sorted(pools[qid], key=lambda d: -bench_qrels[qid][d])
    for d in ranked[:6] + [x for x in ranked if bench_qrels[qid][x] == 0][:3]:
        r = by_id[d]
        print(f"  [rel={bench_qrels[qid][d]}] {short(r['title'], 80)}\n          {short(r['description'])}")

### Save the benchmark

* `queries.tsv` – `qid<TAB>query`
* `qrels.tsv` – `qid<TAB>doc_id<TAB>rel` with rel ∈ {0,1,2} (levels below `rel_min` are 0)
* `pools.json` – qid → candidate doc ids (each query is evaluated only against its own pool)
* `docs.jsonl` – metadata + **original description** for every candidate (lets notebook 02 run before any table is read)
* `needed_files.tsv` / `needed_files.txt` – every CSV file of every candidate (all policies need them; `first`
  falls back to the next CSV if the first one can't be acquired). The `.txt` holds archive member paths, usable with
  `tar -x -T needed_files.txt`.

In [ ]:
def license_of(rec):
    m = re.search(r"is licensed under (.+?)(?: \(http|$)", rec.get("attribution") or "")
    return m.group(1).strip() if m else None


prev_cfg = json.loads((BENCH / "config.json").read_text()) if (BENCH / "config.json").exists() else None
if prev_cfg and prev_cfg.get("chosen") != CHOSEN:
    print(f"NOTE: the configuration changed since the last run ({prev_cfg.get('chosen')} -> {CHOSEN}). "
          "Already profiled tables are kept; only new candidates will be acquired/profiled.")

pool_docs = sorted({d for q in queries for d in pools[q]})
with open(BENCH / "queries.tsv", "w", encoding="utf-8") as f:
    for qid in sorted(queries):
        f.write(f"{qid}\t{queries[qid]}\n")
with open(BENCH / "qrels.tsv", "w", encoding="utf-8") as f:
    for qid in sorted(queries):
        for d in pools[qid]:
            f.write(f"{qid}\t{d}\t{bench_qrels[qid][d]}\n")
(BENCH / "pools.json").write_text(json.dumps({q: pools[q] for q in sorted(queries)}, indent=1))

needed = []  # (doc_id, data_filename, data_url, data_format, position among the dataset's CSV files)
with open(BENCH / "docs.jsonl", "w", encoding="utf-8") as f:
    for d in pool_docs:
        r = by_id[d]
        files = csv_files(r, CHOSEN["csv_mode"])
        for k, fd in enumerate(files):
            needed.append((d, fd["data_filename"], fd.get("data_url"), fd.get("data_format"), k))
        f.write(json.dumps({
            "id": d, "title": r.get("title"), "description": r.get("description"), "url": r.get("url"),
            "publisher": (r.get("data_fields") or {}).get("Publisher"), "license": license_of(r),
            "attribution": r.get("attribution"), "n_files": len(r.get("data") or []),
            "csv_files": [{"data_filename": fd["data_filename"], "data_url": fd.get("data_url"),
                           "data_format": fd.get("data_format")} for fd in files]}, ensure_ascii=False) + "\n")
needed_df = pd.DataFrame(needed, columns=["doc_id", "data_filename", "data_url", "data_format", "csv_index"])
needed_df["member"] = "data_search_e_data/" + needed_df.data_filename
needed_df.to_csv(BENCH / "needed_files.tsv", sep="\t", index=False)
(BENCH / "needed_files.txt").write_text("\n".join(sorted(set(needed_df.member))) + "\n")
(BENCH / "config.json").write_text(json.dumps({
    "chosen": CHOSEN, "auto_selected": AUTO_SELECT_CONFIG, "closest_to_paper": best_cfg,
    "min_relevant": MIN_RELEVANT, "min_candidates": MIN_CANDIDATES, "multi_file_policy": MULTI_FILE_POLICY,
    "stats": stats, "paper": PAPER, "created": datetime.now(timezone.utc).isoformat(),
    "fixture_mode": FIXTURE_MODE}, indent=1))
per_doc = needed_df.groupby("doc_id").size()
print(f"saved {len(queries)} queries, {sum(len(p) for p in pools.values())} qrels rows, {len(pool_docs)} unique docs")
print(f"needed CSV files: {len(needed_df)} ({needed_df.data_filename.nunique()} unique); "
      f"datasets with >1 CSV file: {(per_doc > 1).sum()} (max {per_doc.max()})")

## 4. Table acquisition

Two ways to get the files:

* **A. Source URLs** – every file has a `data_url`. Many still work (we measured ≈ 50 % on a sample in Nov 2026), but
  a live portal serves *today's* data, not the 2019 snapshot that was judged. `data_filename` is `sha256(data_url)`,
  not a content hash, so a download cannot be verified against the snapshot.
* **B. Stream the 478 GB archive once.** It is a single `tar.bz2` holding `data_search_e_data/<data_filename>` in no
  particular order, so it can only be read front to back. We pipe it through `lbzip2 -dc` (all CPU cores) and read the
  tar stream in Python, writing only the needed members to `/content/raw/archive`. This is the equivalent of
  `curl -L <url> | pv | lbzip2 -dc | tar -x -T needed_files.txt`, but with progress, a manifest and early stop.

**Disconnects.** The download feeder resumes with an HTTP `Range` request if the *network* drops, so a broken
connection costs nothing. If the *Colab runtime* dies, the decompressor state is lost and the pass restarts from byte 0;
files already extracted are skipped (manifest), but the stream itself must be read again from the start. With
bzip2 on Colab's 2 vCPUs the full pass takes several hours (the peek below measures it). Use Colab Pro / a
background-execution runtime, keep the tab open, or run this section on any Linux box: it only needs a few GB of disk.

**Default (`ACQUISITION_STRATEGY="auto"`, `ARCHIVE_EXTRACT="all"`)**: A runs first (fast, gives a usable copy for
pilots); then B extracts *every* needed file from the archive. Since B must read the whole archive anyway once any file
is missing, taking all files from the 2019 snapshot costs nothing extra. Profiling prefers the archive copy and
records which source was used. Set `ARCHIVE_EXTRACT="missing"` to use B only for what A could not get.

In [ ]:
MANIFEST = ACQ / "manifest.jsonl"
ARCHIVE_STATE = ACQ / "archive_pass.json"  # written after a complete pass: files not present in the archive
TABLES = INTER / "tables.jsonl"


def load_manifest():
    m = {}
    for row in read_jsonl(MANIFEST):
        m[(row["data_filename"], row["source"])] = row
    return m


def local_copy(fn, source):
    """Path of a usable local copy (exists on this machine and matches the manifest size), else None."""
    row = MANIFEST_ROWS.get((fn, source))
    p = RAW / source / fn
    if row and p.exists() and p.stat().st_size == row["bytes"]:
        return p
    return None


def load_tables():
    last = {}
    for row in read_jsonl(TABLES):
        last[row["id"]] = row
    return last


def needs_profiling(row):
    if row is None:
        return True
    err = row.get("error")
    if not err:
        return False
    return err.startswith("not_acquired") or RETRY_FAILED_TABLES


MANIFEST_ROWS = load_manifest()
done_tables = load_tables()
todo_docs = {d for d in pool_docs if needs_profiling(done_tables.get(d))}
targets = sorted(set(needed_df[needed_df.doc_id.isin(todo_docs)].data_filename))
url_of = dict(zip(needed_df.data_filename, needed_df.data_url))
archive_state = json.loads(ARCHIVE_STATE.read_text()) if ARCHIVE_STATE.exists() else {}
not_in_archive = set(archive_state.get("not_found", []))
print(f"docs still to profile: {len(todo_docs)} / {len(pool_docs)};  files they need: {len(targets)}")
print(f"  already local: archive copy {sum(bool(local_copy(f, 'archive')) for f in targets)}, "
      f"URL copy {sum(bool(local_copy(f, 'url')) for f in targets)};  known absent from archive: "
      f"{len(not_in_archive & set(targets))}")

### 4A. Source URLs: liveness check on a sample, then download

In [ ]:
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "Mozilla/5.0 (compatible; ntcir-ddg-reproduction/1.0)"


def looks_html(head: bytes) -> bool:
    h = head.lstrip()[:600].lower()
    return h.startswith(b"<!doctype html") or h.startswith(b"<html") or b"<html" in h[:300]


def check_url(url):
    try:
        with SESSION.get(url, stream=True, timeout=(10, URL_TIMEOUT), allow_redirects=True) as r:
            if r.status_code != 200:
                return f"http_{r.status_code}"
            head = next(r.iter_content(2048), b"")
            return "html" if looks_html(head) else "alive"
    except requests.RequestException as e:
        return type(e).__name__


def download_url(fn, url):
    dst, tmp = RAW / "url" / fn, RAW / "url" / (fn + ".part")
    err = None
    for attempt in range(URL_RETRIES):
        try:
            with SESSION.get(url, stream=True, timeout=(10, URL_TIMEOUT), allow_redirects=True) as r:
                if r.status_code != 200:
                    err = f"http_{r.status_code}"
                    if 400 <= r.status_code < 500 and r.status_code != 429:
                        break  # permanent
                    raise requests.HTTPError(err)
                h, n, first = hashlib.sha256(), 0, True
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        if first:
                            first = False
                            if looks_html(chunk):
                                err = "html_response"
                                break
                        n += len(chunk)
                        if n > URL_MAX_BYTES:
                            err = "too_large"
                            break
                        h.update(chunk)
                        f.write(chunk)
                if err in ("html_response", "too_large"):
                    tmp.unlink(missing_ok=True)
                    break
                if n == 0:
                    err = "empty_response"
                    tmp.unlink(missing_ok=True)
                    break
                os.replace(tmp, dst)
                return {"data_filename": fn, "source": "url", "bytes": n, "sha256": h.hexdigest(), "url": url,
                        "at": datetime.now(timezone.utc).isoformat()}, None
        except requests.RequestException as e:
            err = err if (err or "").startswith("http_") else type(e).__name__
            time.sleep(min(2 ** attempt, 10))
    tmp.unlink(missing_ok=True)
    return None, err


RUN_URL_PASS = ACQUISITION_STRATEGY in ("auto", "url") and HAS_SOURCE_URLS
url_targets = [f for f in targets if url_of.get(f) and not local_copy(f, "url") and not local_copy(f, "archive")]
if not RUN_URL_PASS:
    print("URL pass skipped (strategy or no URL field)")
elif not url_targets:
    print("URL pass: nothing to download")
else:
    sample = random.Random(0).sample(url_targets, min(URL_CHECK_SAMPLE, len(url_targets)))
    with ThreadPoolExecutor(URL_WORKERS) as ex:
        status = list(ex.map(lambda f: check_url(url_of[f]), sample))
    c = collections.Counter(status)
    print(f"liveness on {len(sample)} sampled URLs: alive {c['alive'] / len(sample):.0%}  ->  {dict(c)}")
    domains = collections.Counter(url_of[f].split("/")[2] for f in sample if f)
    print("sampled domains:", dict(domains.most_common(8)))

    log(f"downloading {len(url_targets)} files from their source URLs ...")
    failures, ok = [], 0
    with ThreadPoolExecutor(URL_WORKERS) as ex:
        futs = {ex.submit(download_url, f, url_of[f]): f for f in url_targets}
        for i, fut in enumerate(as_completed(futs), 1):
            row, err = fut.result()
            if row:
                append_jsonl(MANIFEST, row)
                ok += 1
            else:
                failures.append({"data_filename": futs[fut], "url": url_of[futs[fut]], "error": err})
            if i % 50 == 0 or i == len(futs):
                log(f"  {i}/{len(futs)} done, {ok} ok, {len(failures)} failed")
    print("failure reasons (this run):", dict(collections.Counter(f["error"] for f in failures)))
    MANIFEST_ROWS = load_manifest()
    # cumulative failure log across runs; files acquired since are dropped from it
    fail_path = ACQ / "url_failures.tsv"
    old = pd.read_csv(fail_path, sep="\t", dtype=str) if fail_path.exists() else pd.DataFrame(columns=["data_filename", "url", "error"])
    merged = pd.concat([old, pd.DataFrame(failures, columns=["data_filename", "url", "error"])]).drop_duplicates(
        "data_filename", keep="last")
    merged = merged[[not local_copy(f, "url") for f in merged.data_filename]]
    merged.to_csv(fail_path, sep="\t", index=False)
    print(f"url_failures.tsv: {len(merged)} files, reasons: {merged.error.value_counts().to_dict()}")

### 4B. Stream the archive
First a **peek**: read only the first few hundred members to confirm the layout (`data_search_e_data/<data_filename>`),
check that member names map onto `data_filename`, and measure throughput to estimate the full pass.

In [ ]:
class ArchiveStream:
    """Source bytes -> `lbzip2 -dc` subprocess -> tarfile 'r|' in this process.

    A feeder thread copies the compressed bytes into the decompressor. If the HTTP connection drops it reconnects
    with `Range: bytes=<offset>-`, so the decompressor never notices. `simulate_drop_at` (fixture tests only) raises
    one fake connection error at that byte offset to exercise the resume path.
    """

    def __init__(self, url=None, path=None, simulate_drop_at=None, max_reconnects=50):
        self.url, self.path, self.simulate_drop_at = url, path, simulate_drop_at
        self.max_reconnects, self.bytes_in, self.reconnects, self.error = max_reconnects, 0, 0, None
        self._stop = threading.Event()

    def __enter__(self):
        self.proc = subprocess.Popen([DECOMPRESSOR, "-dc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                     stderr=subprocess.DEVNULL, bufsize=0)
        self.thread = threading.Thread(target=self._feed, daemon=True)
        self.thread.start()
        self.started = time.time()
        return self

    def _chunks(self, offset):
        if self.path:
            with open(self.path, "rb") as f:
                f.seek(offset)
                while chunk := f.read(1 << 16):
                    yield chunk
            return
        headers = {"Range": f"bytes={offset}-"} if offset else {}
        if HF_TOKEN:
            headers["Authorization"] = f"Bearer {HF_TOKEN}"
        with requests.get(self.url, headers=headers, stream=True, timeout=(30, 300)) as r:
            if offset and r.status_code != 206:
                raise RuntimeError(f"server ignored the Range request (HTTP {r.status_code}); cannot resume")
            r.raise_for_status()
            yield from r.iter_content(8 << 20)

    def _feed(self):
        dropped = False
        while not self._stop.is_set():
            try:
                for chunk in self._chunks(self.bytes_in):
                    if self._stop.is_set():
                        break
                    if self.simulate_drop_at and not dropped and self.bytes_in + len(chunk) > self.simulate_drop_at:
                        dropped = True
                        raise requests.ConnectionError("simulated connection drop")
                    self.proc.stdin.write(chunk)
                    self.bytes_in += len(chunk)
                break  # EOF
            except requests.RequestException:
                pass  # network problem: reconnect below and resume at self.bytes_in
            except (BrokenPipeError, ValueError, OSError):
                break  # decompressor closed (early stop)
            except Exception as e:  # noqa: BLE001
                self.error = repr(e)
                break
            self.reconnects += 1
            if self.reconnects > self.max_reconnects:
                self.error = "too many reconnects"
                break
            log(f"  connection lost at {self.bytes_in / 1e9:.2f} GB; resuming with a Range request "
                f"(reconnect {self.reconnects})")
            time.sleep(min(2 ** min(self.reconnects, 6), 60))
        try:
            self.proc.stdin.close()
        except OSError:
            pass

    def __exit__(self, *exc):
        self._stop.set()
        try:
            self.proc.stdout.close()
        except OSError:
            pass
        self.proc.kill()
        self.proc.wait()
        self.thread.join(timeout=5)

    def rate(self):
        return self.bytes_in / max(time.time() - self.started, 1e-6)


def open_archive(**kw):
    if FIXTURE_MODE:
        return ArchiveStream(path=ARCHIVE_PATH, **kw)
    return ArchiveStream(url=ARCHIVE_URL, **kw)


def peek_archive(n_members):
    names = []
    with open_archive() as st:
        tf = tarfile.open(fileobj=st.proc.stdout, mode="r|", bufsize=1 << 20)
        for m in tf:
            names.append((m.name, m.size, m.type))
            if len(names) >= n_members:
                break
        elapsed, rate, nbytes = time.time() - st.started, st.rate(), st.bytes_in
    return names, elapsed, rate, nbytes


RUN_ARCHIVE_PASS = ACQUISITION_STRATEGY in ("auto", "archive")
if ARCHIVE_EXTRACT == "all":
    archive_targets = [f for f in targets if not local_copy(f, "archive") and f not in not_in_archive]
else:
    archive_targets = [f for f in targets if not local_copy(f, "archive") and not local_copy(f, "url")
                       and f not in not_in_archive]
print(f"files to take from the archive: {len(archive_targets)}")

if RUN_ARCHIVE_PASS and archive_targets:
    log(f"peeking at the first {PEEK_MEMBERS} members ...")
    names, elapsed, rate, nbytes = peek_archive(PEEK_MEMBERS)
    print(f"read {nbytes / 1e6:.1f} MB compressed in {elapsed:.1f}s -> {rate / 1e6:.1f} MB/s")
    for n, size, typ in names[:8]:
        print(f"  {'d' if typ == tarfile.DIRTYPE else '-'} {size:>12,d}  {n}")
    files_only = [(n, s) for n, s, t in names if t != tarfile.DIRTYPE]
    prefixes = collections.Counter(n.rsplit("/", 1)[0] + "/" for n, _ in files_only)
    print("member prefixes:", dict(prefixes))
    all_filenames = {d["data_filename"] for r in collection for d in r["data"]}
    hit = sum(n.rsplit("/", 1)[-1] in all_filenames for n, _ in files_only)
    print(f"peeked member basenames that are a collection data_filename: {hit}/{len(files_only)}"
          "  -> mapping rule: member = 'data_search_e_data/' + data_filename")
    est_h = ARCHIVE_SIZE / max(rate, 1) / 3600
    n_t = len(archive_targets)
    print(f"\nESTIMATE for a full pass over {ARCHIVE_SIZE / 1e9:.0f} GB at {rate / 1e6:.1f} MB/s: {est_h:.1f} h "
          "(the peek includes connection start-up; the steady rate is usually a bit higher).")
    print(f"Members are in random order, so the last of {n_t} needed files is expected at ~{n_t / (n_t + 1):.1%} of "
          "the stream: early stop saves little.")
else:
    print("archive pass not needed" if RUN_ARCHIVE_PASS else "archive pass disabled by ACQUISITION_STRATEGY")

### The full pass
Prints progress every 60 s (compressed GB read, members seen, needed files found, ETA). Stops early once every needed
file is found. Each extracted file is written to `*.part`, size-checked against the tar header, renamed, and logged in
the manifest, so an interrupted pass never leaves a half-written file that looks complete.

In [ ]:
def stream_extract(wanted, progress_every=60, simulate_drop_at=None):
    remaining = set(wanted)
    seen = found = 0
    dec_bytes = 0
    last = time.time()
    complete = False
    with open_archive(simulate_drop_at=simulate_drop_at) as st:
        tf = tarfile.open(fileobj=st.proc.stdout, mode="r|", bufsize=1 << 20)
        try:
            for m in tf:
                seen += 1
                dec_bytes += m.size
                base = m.name.rsplit("/", 1)[-1]
                if m.isfile() and base in remaining:
                    dst, tmp = RAW / "archive" / base, RAW / "archive" / (base + ".part")
                    src = tf.extractfile(m)
                    h, n = hashlib.sha256(), 0
                    with open(tmp, "wb") as f:
                        while chunk := src.read(1 << 20):
                            h.update(chunk)
                            n += len(chunk)
                            f.write(chunk)
                    if n != m.size:
                        raise IOError(f"short read for {base}: {n} != {m.size}")
                    os.replace(tmp, dst)
                    append_jsonl(MANIFEST, {"data_filename": base, "source": "archive", "bytes": n,
                                            "sha256": h.hexdigest(), "member": m.name,
                                            "at": datetime.now(timezone.utc).isoformat()})
                    remaining.discard(base)
                    found += 1
                    if not remaining:
                        log("all needed files found - stopping early")
                        break
                if time.time() - last > progress_every:
                    last = time.time()
                    frac = st.bytes_in / ARCHIVE_SIZE
                    eta = (ARCHIVE_SIZE - st.bytes_in) / max(st.rate(), 1) / 3600
                    log(f"  {st.bytes_in / 1e9:8.1f} GB ({frac:5.1%}) | {st.rate() / 1e6:5.1f} MB/s | members {seen:,d} | "
                        f"found {found}/{len(wanted)} | ETA {eta:.1f} h | reconnects {st.reconnects}")
            else:
                complete = st.error is None
        except tarfile.ReadError as e:
            raise RuntimeError(f"tar stream ended unexpectedly at {st.bytes_in / 1e9:.2f} GB: {e} "
                               f"(feeder error: {st.error})") from e
        if st.error:
            raise RuntimeError(f"download failed: {st.error}")
        info = {"bytes_read": st.bytes_in, "members_seen": seen, "decompressed_bytes": dec_bytes,
                "seconds": time.time() - st.started, "reconnects": st.reconnects}
    return found, remaining, complete, info


if RUN_ARCHIVE_PASS and archive_targets:
    if not CONFIRM_ARCHIVE_STREAM:
        print(f"{len(archive_targets)} files must come from the archive. Read the estimate above, then set "
              "CONFIRM_ARCHIVE_STREAM = True in the config cell and re-run this cell (or Run all).")
    else:
        log(f"streaming the archive for {len(archive_targets)} files ...")
        found, remaining, complete, info = stream_extract(
            archive_targets, progress_every=1 if FIXTURE_MODE else 60,
            simulate_drop_at=30_000 if FIXTURE_MODE else None)
        log(f"done: found {found}, not found {len(remaining)}, complete pass: {complete}, info: {info}")
        if complete:
            ARCHIVE_STATE.write_text(json.dumps({"not_found": sorted(remaining | not_in_archive),
                                                 "completed": datetime.now(timezone.utc).isoformat(), **info}, indent=1))
        MANIFEST_ROWS = load_manifest()

### Acquisition summary

In [ ]:
MANIFEST_ROWS = load_manifest()
acq_rows = []
for f in sorted(set(needed_df.data_filename)):
    src = "archive" if local_copy(f, "archive") else "url" if local_copy(f, "url") else None
    acq_rows.append({"data_filename": f, "local_source": src})
acq_df = pd.DataFrame(acq_rows)
print("needed files by local source (only files whose doc still needs profiling matter):")
print(acq_df.local_source.fillna("missing").value_counts().to_string())
if not_in_archive:
    print(f"known to be absent from the archive: {len(not_in_archive)}")

## 5. Per-table preprocessing (no LLM)

For each candidate dataset:
1. pick the file(s) per `MULTI_FILE_POLICY`, preferring the archive copy;
2. **sniff**: reject HTML pages / binary / empty files; encoding = `utf-8-sig` if there is a BOM, else `utf-8` if the
   whole file decodes, else `latin-1`; delimiter = the one among `, ; TAB |` that gives the most consistent column
   count over the first lines; gzip/zip are opened transparently;
3. **read**: files ≤ `LARGE_FILE_BYTES` are read whole (`on_bad_lines="warn"`, skipped lines are counted) and
   profiled with `autoddg.profiling.profile_dataset(df)` (this is exactly what `AutoDDG.profile_dataframe` calls).
   Larger files are streamed in chunks: exact row count + uniform reservoir sample of `PROFILE_SAMPLE_ROWS` rows, then
   `summarize(profile_metadata(sample, nb_rows=true_count))`, so the profile reports the full row count;
4. the LLM sample is `autoddg.utils.get_sample(df, sample_size=100)` (default `random_state=9`), as in the
   quick-start notebook. For streamed files it is drawn from the reservoir sample.

Output: one JSON line per table in `intermediate/tables.jsonl` (appended and fsync'ed after each table; a re-run
skips tables that already succeeded).

In [ ]:
class TableError(Exception):
    def __init__(self, code, msg=""):
        super().__init__(f"{code}: {msg}" if msg else code)
        self.code = code


def open_binary(path):
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic[:2] == b"\x1f\x8b":
        return gzip.open(path, "rb"), "gzip"
    if magic == b"PK\x03\x04":
        z = zipfile.ZipFile(path)
        members = [i for i in z.infolist() if not i.is_dir()]
        if not members:
            raise TableError("empty_zip")
        csvs = [i for i in members if i.filename.lower().endswith((".csv", ".txt"))] or members
        pick = max(csvs, key=lambda i: i.file_size)
        return z.open(pick), f"zip:{pick.filename}"
    return open(path, "rb"), None


def utf8_ok(path):
    dec = codecs.getincrementaldecoder("utf-8")()
    fh, _ = open_binary(path)
    with fh:
        try:
            while chunk := fh.read(1 << 22):
                dec.decode(chunk)
            dec.decode(b"", final=True)
            return True
        except UnicodeDecodeError:
            return False


def pick_delimiter(text):
    best, best_score = ",", None
    for d in [",", ";", "\t", "|"]:
        try:
            counts = [len(r) for r in csv.reader(io.StringIO(text), delimiter=d)][:-1] or [1]
        except csv.Error:
            continue
        mode, freq = collections.Counter(counts).most_common(1)[0]
        score = (mode > 1, freq / len(counts), mode)
        if best_score is None or score > best_score:
            best, best_score = d, score
    return best


def sniff(path):
    size = path.stat().st_size
    fh, comp = open_binary(path)
    with fh:
        head = fh.read(1 << 20)
    if not head.strip():
        raise TableError("empty", "file is empty")
    if looks_html(head):
        raise TableError("html", "file is an HTML page, not a CSV")
    if b"\x00" in head[:8192]:
        raise TableError("binary", "file contains NUL bytes")
    if head.startswith(codecs.BOM_UTF8):
        enc = "utf-8-sig"
    else:
        enc = "utf-8" if utf8_ok(path) else "latin-1"
    text = head.decode(enc, errors="replace")
    if len(head) == (1 << 20) and "\n" in text:
        text = text[: text.rfind("\n")]
    text = "\n".join(text.splitlines()[:200])
    return {"path": path, "bytes": size, "compression": comp, "encoding": enc, "delimiter": pick_delimiter(text)}


def count_bad_lines(caught):
    return sum(str(w.message).count("Skipping line") for w in caught)


def read_small(info):
    fh, _ = open_binary(info["path"])
    with fh, warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        df = pd.read_csv(fh, sep=info["delimiter"], encoding=info["encoding"], on_bad_lines="warn", low_memory=False)
    return df, count_bad_lines(caught)


def read_streaming(infos, k, chunk_rows, seed):
    """Exact row count + uniform sample of k rows (random keys, keep the k smallest) over one or more files."""
    rng = np.random.default_rng(seed)
    keep, total, bad, header = None, 0, 0, None
    for info in infos:
        fh, _ = open_binary(info["path"])
        with fh, warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            for chunk in pd.read_csv(fh, sep=info["delimiter"], encoding=info["encoding"], dtype=str,
                                     keep_default_na=False, on_bad_lines="warn", chunksize=chunk_rows):
                if header is None:
                    header = list(chunk.columns)
                chunk.index = pd.RangeIndex(total, total + len(chunk))
                total += len(chunk)
                chunk["__key__"] = rng.random(len(chunk))
                keep = chunk if keep is None else pd.concat([keep, chunk])
                if len(keep) > k:
                    keep = keep.nsmallest(k, "__key__")
            bad += count_bad_lines(caught)
    sample = keep.sort_index().drop(columns="__key__")
    # re-infer dtypes exactly as a normal pd.read_csv of the file would
    df = pd.read_csv(io.StringIO(sample.to_csv(index=False)), low_memory=False)
    return df, total, bad


def choose_files(doc_id):
    """Return [(data_filename, source, path)] per MULTI_FILE_POLICY, preferring archive copies."""
    avail = []
    for fn in needed_df[needed_df.doc_id == doc_id].sort_values("csv_index").data_filename:
        for source in ("archive", "url"):
            p = local_copy(fn, source)
            if p:
                avail.append((fn, source, p))
                break
    if not avail:
        return []
    if MULTI_FILE_POLICY == "largest":
        return [max(avail, key=lambda a: a[2].stat().st_size)]
    if MULTI_FILE_POLICY == "concat":
        return avail
    return avail[:1]


def profile_doc(doc_id):
    rec = by_id[doc_id]
    row = {"id": doc_id, "title": rec.get("title"), "original_description": rec.get("description"),
           "url": rec.get("url"), "publisher": (rec.get("data_fields") or {}).get("Publisher"),
           "license": license_of(rec), "n_files": len(rec.get("data") or []),
           "n_csv_files": int((needed_df.doc_id == doc_id).sum()), "file_used": None, "file_source": None,
           "file_bytes": None, "encoding": None, "delimiter": None, "n_rows": None, "n_cols": None, "columns": None,
           "streamed": None, "profiled_rows": None, "sample_csv": None, "basic_profile": None,
           "structural_profile": None, "read_notes": [], "error": None, "seconds": None,
           "multi_file_policy": MULTI_FILE_POLICY, "autoddg_version": autoddg.__version__,
           "profiled_at": datetime.now(timezone.utc).isoformat()}
    t0 = time.time()
    notes = row["read_notes"]
    try:
        chosen = choose_files(doc_id)
        if not chosen:
            raise TableError("not_acquired", "no local copy of any CSV file of this dataset")
        infos = []
        for fn, source, path in chosen:
            try:
                infos.append({**sniff(path), "data_filename": fn, "source": source})
            except TableError as e:
                if not infos and len(chosen) == 1:
                    raise
                notes.append(f"skipped {fn[:12]}…: {e}")
        if not infos:
            raise TableError("no_readable_file", "; ".join(notes))
        if len(chosen) > 1 or row["n_csv_files"] > 1:
            notes.append(f"{row['n_csv_files']} CSV files; policy={MULTI_FILE_POLICY}; using {len(infos)}")
        if any(i["source"] == "url" for i in infos):
            notes.append("copy from the live source URL (may differ from the 2019 snapshot)")
        enc = {i["encoding"] for i in infos}
        delim = {i["delimiter"] for i in infos}
        if enc != {"utf-8"}:
            notes.append(f"encoding={'/'.join(sorted(enc))}")
        if delim != {","}:
            notes.append(f"delimiter={'/'.join(repr(d) for d in sorted(delim))}")
        if any(i["compression"] for i in infos):
            notes.append(f"compressed: {[i['compression'] for i in infos]}")

        total_bytes = sum(i["bytes"] for i in infos)
        if total_bytes <= LARGE_FILE_BYTES:
            frames, bad = [], 0
            for info in infos:
                df_i, b = read_small(info)
                bad += b
                if frames and list(df_i.columns) != list(frames[0].columns):
                    notes.append(f"skipped {info['data_filename'][:12]}…: header differs")
                    continue
                frames.append(df_i)
            df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
            n_rows, streamed = len(df), False
        else:
            df, n_rows, bad = read_streaming(infos, PROFILE_SAMPLE_ROWS, READ_CHUNK_ROWS, RESERVOIR_SEED)
            streamed = True
            notes.append(f"streamed {total_bytes / 1e6:.1f} MB: exact row count, profiled a {len(df):,d}-row sample")
        if bad:
            notes.append(f"skipped {bad} malformed line(s)")
        if n_rows == 0:
            raise TableError("no_rows", "header only")
        row.update(file_used=";".join(i["data_filename"] for i in infos),
                   file_source=";".join(i["source"] for i in infos), file_bytes=total_bytes,
                   encoding="/".join(sorted(enc)), delimiter="/".join(sorted(delim)), n_rows=int(n_rows),
                   n_cols=int(df.shape[1]), columns=[str(c) for c in df.columns], streamed=streamed)
        _, row["sample_csv"] = get_sample(df, sample_size=SAMPLE_SIZE, random_state=SAMPLE_SEED)
        try:
            if streamed:
                meta = profile_metadata(df, nb_rows=int(n_rows))
                row["basic_profile"], row["structural_profile"] = summarize(meta)
                row["profiled_rows"] = meta.get("nb_profiled_rows")
            else:
                row["basic_profile"], row["structural_profile"] = profile_dataset(df)
        except Exception as e:  # noqa: BLE001
            raise TableError("profile_failed", repr(e)[:300]) from e
    except TableError as e:
        row["error"] = str(e)
    except Exception as e:  # noqa: BLE001
        row["error"] = f"read_failed: {repr(e)[:300]}"
    row["seconds"] = round(time.time() - t0, 2)
    return row


done_tables = load_tables()
todo = [d for d in pool_docs if needs_profiling(done_tables.get(d))]
log(f"profiling {len(todo)} tables ({len(pool_docs) - len(todo)} already done)")
t_start, last = time.time(), time.time()
for i, doc_id in enumerate(todo, 1):
    row = profile_doc(doc_id)
    append_jsonl(TABLES, row)
    if DELETE_RAW_AFTER_PROFILE and not row["error"]:
        for fn in (row["file_used"] or "").split(";"):
            for source in ("archive", "url"):
                (RAW / source / fn).unlink(missing_ok=True)
    if time.time() - last > 30 or i == len(todo):
        last = time.time()
        log(f"  {i}/{len(todo)}  ({(time.time() - t_start) / i:.1f}s/table)  last: "
            f"{row['n_rows']} rows x {row['n_cols']} cols, error={row['error']}")
tables = load_tables()
print(f"tables.jsonl: {len(tables)} tables, {sum(not r['error'] for r in tables.values())} ok")

## 6. Statistics of the acquired tables vs the paper

In [ ]:
tables = load_tables()
ok = {i: r for i, r in tables.items() if not r["error"] and i in set(pool_docs)}
rows_s = pd.Series([r["n_rows"] for r in ok.values()], dtype=float)
cols_s = pd.Series([r["n_cols"] for r in ok.values()], dtype=float)
summary = pd.DataFrame({
    "ours": [len(ok), rows_s.mean(), rows_s.median(), rows_s.min(), rows_s.max(), cols_s.mean(), cols_s.median(),
             cols_s.min(), cols_s.max()],
    "paper": [PAPER["tables_total"], PAPER["rows_avg"], None, PAPER["rows_min"], PAPER["rows_max"], PAPER["cols_avg"],
              None, PAPER["cols_min"], PAPER["cols_max"]]},
    index=["tables ok (unique; paper: 615 summed over queries)", "rows avg", "rows median", "rows min", "rows max", "cols avg", "cols median",
           "cols min", "cols max"])
display(summary.round(1))

import matplotlib.pyplot as plt  # noqa: E402

if len(ok):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].hist(np.log10(rows_s.clip(lower=1)), bins=30)
    ax[0].set_xlabel("log10(rows)")
    ax[0].set_ylabel("tables")
    ax[0].axvline(np.log10(PAPER["rows_avg"]), ls="--", c="k", label="paper mean")
    ax[0].legend()
    ax[1].hist(cols_s, bins=30)
    ax[1].set_xlabel("columns")
    ax[1].axvline(PAPER["cols_avg"], ls="--", c="k", label="paper mean")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

failed = {i: r for i, r in tables.items() if r["error"] and i in set(pool_docs)}
print(f"failures: {len(failed)}")
print(pd.Series(collections.Counter(r["error"].split(":")[0] for r in failed.values())).to_string()
      if failed else "  none")
if failed:
    display(pd.DataFrame([{"id": i, "title": short(r["title"], 60), "error": short(r["error"], 100)}
                          for i, r in list(failed.items())[:30]]))
notes = collections.Counter(n.split(":")[0].split("=")[0] for r in ok.values() for n in r["read_notes"])
print("read notes on successful tables:", dict(notes))

### Coverage per query
A query whose candidates failed may drop below the filter (≥ 5 relevant, ≥ 10 candidates). Those queries are dropped
from the **acquired** benchmark (`queries_acquired.tsv`, `qrels_acquired.tsv`, `pools_acquired.json`), which is what
notebook 02 uses for the UFD/SFD comparison so every variant is scored on the same documents.

In [ ]:
cov = []
for qid in sorted(queries):
    pool = pools[qid]
    ok_pool = [d for d in pool if d in ok]
    rel = [d for d in pool if bench_qrels[qid][d] > 0]
    ok_rel = [d for d in rel if d in ok]
    cov.append({"qid": qid, "query": queries[qid], "candidates": len(pool), "candidates_ok": len(ok_pool),
                "relevant": len(rel), "relevant_ok": len(ok_rel),
                "keep": len(ok_rel) >= MIN_RELEVANT and len(ok_pool) >= MIN_CANDIDATES})
cov_df = pd.DataFrame(cov)
cov_df.to_csv(BENCH / "coverage.tsv", sep="\t", index=False)
display(cov_df)
kept = cov_df[cov_df.keep].qid.tolist()
pools_acq = {q: [d for d in pools[q] if d in ok] for q in kept}
with open(BENCH / "queries_acquired.tsv", "w", encoding="utf-8") as f:
    for q in kept:
        f.write(f"{q}\t{queries[q]}\n")
with open(BENCH / "qrels_acquired.tsv", "w", encoding="utf-8") as f:
    for q in kept:
        for d in pools_acq[q]:
            f.write(f"{q}\t{d}\t{bench_qrels[q][d]}\n")
(BENCH / "pools_acquired.json").write_text(json.dumps(pools_acq, indent=1))
s_acq = bench_stats({q: queries[q] for q in kept}, {q: bench_qrels[q] for q in kept}, pools_acq)
print(f"queries kept after acquisition: {len(kept)}/{len(queries)}")
display(pd.DataFrame([{"": "acquired", **{k: round(v, 2) if isinstance(v, float) else v for k, v in s_acq.items()}},
                      {"": "planned", **{k: round(v, 2) if isinstance(v, float) else v for k, v in stats.items()}},
                      {"": "paper", "n_queries": 32, "rel_avg": 10, "rel_min": 5, "rel_max": 22, "cand_avg": 19,
                       "cand_min": 10, "cand_max": 42, "tables_sum": 615}]).set_index(""))

### Three sample tables

In [ ]:
for i in random.Random(1).sample(sorted(ok), min(3, len(ok))):
    r = ok[i]
    print("=" * 110)
    print(f"{r['title']}  [{r['n_rows']:,d} rows x {r['n_cols']} cols, source={r['file_source']}, notes={r['read_notes']}]")
    print("ORIGINAL DESCRIPTION:", short(r["original_description"], 600))
    display(pd.read_csv(io.StringIO(r["sample_csv"]), dtype=str, keep_default_na=False).head())
    print(r["basic_profile"][:1500])
    print(r["structural_profile"][:600] or "(no temporal/spatial coverage detected)")

## 7. Package for the Mac
`benchmark/` + `intermediate/` zipped to `WORK_DIR/ntcir_ddg_intermediate.zip` (should be a few MB). Download it from
Drive to the Mac and follow `mac/README.md`.

In [ ]:
ZIP_PATH = WORK / "ntcir_ddg_intermediate.zip"
tmp_zip = Path("/tmp") / ZIP_PATH.name
with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (BENCH, INTER):
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(WORK).as_posix())
shutil.copy(tmp_zip, ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH) as z:
    for zi in z.infolist():
        print(f"  {zi.file_size:>12,d}  {zi.filename}")
print(f"\n{ZIP_PATH}  {ZIP_PATH.stat().st_size / 1e6:.2f} MB")